# Preprocessament de dades

En aquest notebook es duu a terme el preprocessament del conjunt de dades de reserves hoteleres, amb l’objectiu de preparar-lo per al desenvolupament posterior de models predictius orientats a estimar el risc de cancel·lació de les reserves.

Les decisions adoptades en aquesta fase es basen en els resultats obtinguts en l’anàlisi exploratòria de dades (EDA). En particular, es tracten els valors absents, s’eliminen variables amb possible *data leakage*, es generen noves característiques derivades i es construeix un pipeline de preprocessament reusable per a les fases d’entrenament, validació i test.

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "Source" else Path.cwd()
assert (REPO_ROOT / "Data").is_dir(), "Open this notebook from the repository root or Source directory."
RUNS_DIR = REPO_ROOT / "runs"
RUNS_DIR.mkdir(exist_ok=True)


In [1]:
import warnings
warnings.filterwarnings("ignore")
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


## 1. Càrrega del conjunt de dades

En primer lloc, es carrega el conjunt de dades original i es prepara l’entorn de treball per aplicar les transformacions posteriors.

In [2]:

DATA_PATH = REPO_ROOT / "Data" / "dataset_5000.csv"
TARGET = "is_canceled"
RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)

print(f"Nombre de files inicial: {df.shape[0]}")
print(f"Nombre de columnes inicial: {df.shape[1]}")
display(df.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset_5000 (1).csv'

## 2. Eliminació de registres duplicats

D’acord amb els resultats de l’EDA, es revisa la presència de registres duplicats i, si escau, s’eliminen per garantir que cada observació representi una reserva única.

In [ ]:
n_duplicates = df.duplicated().sum()
print(f"Nombre de files duplicades abans d'eliminar-les: {n_duplicates}")

df = df.drop_duplicates().copy()

print(f"Nombre de files després d'eliminar duplicats: {df.shape[0]}")

Nombre de files duplicades abans d'eliminar-les: 270
Nombre de files després d'eliminar duplicats: 4730


## 3. Tractament inicial de valors absents

A partir de l’EDA, s’ha observat la presència de valors absents principalment en les variables `children`, `country`, `agent` i `company`. En aquesta fase s’apliquen tractaments inicials amb sentit semàntic, mantenint la coherència amb la naturalesa de cada variable.


In [ ]:
# children: absència interpretada com 0
if "children" in df.columns:
    df["children"] = df["children"].fillna(0)

# country: categoria explícita per valors desconeguts
if "country" in df.columns:
    df["country"] = df["country"].fillna("Unknown")

## 4. Transformació de variables amb molts valors i valors absents

Les variables `agent` i `company` presenten una proporció elevada de valors absents. Tanmateix, en aquest cas els valors nuls no s’interpreten com una absència aleatòria, sinó com la manca d’intermediari o d’empresa associada a la reserva.

Per aquest motiu, en lloc d’imputar un codi numèric artificial, aquestes variables es transformen en indicadors binaris que recullen únicament la presència o absència d’agent i d’empresa.

In [ ]:
# agent i company es transformen en indicadors binaris 0/1
if "agent" in df.columns:
    df["has_agent"] = df["agent"].notna().astype(int)
    df = df.drop(columns=["agent"])

if "company" in df.columns:
    df["has_company"] = df["company"].notna().astype(int)
    df = df.drop(columns=["company"])


# deixar nomes els 10 paisos mes comuns i agrupar la resta com "Other"
if "country" in df.columns:
	top_countries = df["country"].value_counts().nlargest(10).index
	df["country"] = df["country"].where(df["country"].isin(top_countries), other="Other")

display(df[["has_agent", "has_company", "country"]].head())
print(df[["has_agent", "has_company", "country"]].dtypes)

,has_agent,has_company,country
0,1,0,GBR
1,1,0,PRT
2,1,0,BRA
3,1,0,Other
4,1,0,GBR


has_agent       int64
has_company     int64
country        object
dtype: object


## 5. Eliminació de variables amb possible *data leakage*

L’EDA ha posat de manifest que algunes variables contenen informació vinculada al resultat final de la reserva i, per tant, no estarien disponibles en el moment real de fer la predicció. Aquestes variables podrien introduir problemes de *data leakage* i s’eliminen abans de la modelització.

In [ ]:
# Variables clarament problemàtiques o potencialment no disponibles
cols_to_drop = [
    "reservation_status",
    "reservation_status_date",
    "assigned_room_type",
    "booking_changes",
    "days_in_waiting_list",
    "deposit_type"
]

existing_drop_cols = [col for col in cols_to_drop if col in df.columns]
print("Variables eliminades:", existing_drop_cols)

df = df.drop(columns=existing_drop_cols, errors="ignore")

print(f"Nombre de columnes després d'eliminar variables problemàtiques: {df.shape[1]}")#

Variables eliminades: ['reservation_status', 'reservation_status_date', 'assigned_room_type', 'booking_changes', 'days_in_waiting_list', 'deposit_type']
Nombre de columnes després d'eliminar variables problemàtiques: 27


## 6. Eliminació de variables poc informatives (EDA)

In [ ]:
# Variables que no es faran servir al pipeline de ML
ignored_for_ml = ["is_repeated_guest", "distribution_channel", "meal"]
print("Variables ignorades al pipeline de ML:", ignored_for_ml)


Variables ignorades al pipeline de ML: ['is_repeated_guest', 'distribution_channel', 'meal']


## 7. Enginyeria de característiques

Amb l’objectiu d’enriquir la representació de cada reserva, es generen noves variables derivades a partir de les originals. Aquestes variables resumeixen informació rellevant sobre la durada de l’estada, la composició del grup i la relació entre el cost i el nombre d’hostes.

In [ ]:
# Durada total de l'estada
if {"stays_in_weekend_nights", "stays_in_week_nights"}.issubset(df.columns):
    df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]

# Nombre total d'hostes
if {"adults", "children", "babies"}.issubset(df.columns):
    df["total_guests"] = df["adults"] + df["children"] + df["babies"]

# ADR per hoste
if {"adr", "total_guests"}.issubset(df.columns):
    safe_guests = df["total_guests"].replace(0, 1)
    df["adr_per_guest"] = df["adr"] / safe_guests

# Reserva familiar: indicador binari 0/1
if {"children", "babies"}.issubset(df.columns):
    df["is_family_booking"] = ((df["children"] > 0) | (df["babies"] > 0)).astype(int)

# Assignar un cancel_rate per country
country_cancel_rate = df.groupby('country')['is_canceled'].mean()

def risk_bucket(rate):
    if rate > 0.5: return 'High'
    elif rate > 0.3: return 'Medium'
    else: return 'Low'

df['country_risk'] = df['country'].map(country_cancel_rate).apply(risk_bucket)

engineered_features = [
    "total_nights",
    "total_guests",
    "adr_per_guest",
    "is_family_booking",
    "country_risk"
]

display(df[engineered_features].head())

,total_nights,total_guests,adr_per_guest,is_family_booking,country_risk
0,7,2.0,33.40,0,Low
1,3,2.0,38.25,0,High
2,3,3.0,20.00,0,Medium
3,1,1.0,95.00,0,Low
4,4,2.0,54.00,0,Low


In [ ]:
# treure originals i quedar-te amb agregats
cols_to_drop_extra = [
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies"
]

df = df.drop(columns=[col for col in cols_to_drop_extra if col in df.columns])

## 8. Revisió final del conjunt de dades

Abans de separar les dades en conjunts d’entrenament, validació i test, es comprova la dimensió final del dataset i la presència de valors absents en les variables restants.

In [ ]:
print(f"Nombre final de files: {df.shape[0]}")
print(f"Nombre final de columnes: {df.shape[1]}")

missing_final = pd.DataFrame({
    "nuls": df.isna().sum(),
    "percentatge (%)": (df.isna().mean() * 100).round(2)
}).sort_values("percentatge (%)", ascending=False)

display(missing_final[missing_final["nuls"] > 0])

Nombre final de files: 4730
Nombre final de columnes: 27


,nuls,percentatge (%)


## 9. Separació entre variables explicatives i variable objectiu

A continuació, el conjunt de dades es divideix en variables d’entrada (`X`) i variable objectiu (`y`), corresponent a `is_canceled`.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

print("Dimensions de X:", X.shape)
print("Dimensions de y:", y.shape)

Dimensions de X: (4730, 26)
Dimensions de y: (4730,)


## 10. Partició en conjunts d’entrenament, validació i test

Per garantir una avaluació robusta dels models, el conjunt de dades es divideix en tres subconjunts: entrenament, validació i test. La partició es realitza de manera estratificada per mantenir la proporció de la variable objectiu en tots els conjunts.

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train_full
)

print("Dimensions dels conjunts:")
print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")
print()
print(f"y_train: {y_train.shape}")
print(f"y_val:   {y_val.shape}")
print(f"y_test:  {y_test.shape}")

Dimensions dels conjunts:
X_train: (3027, 26)
X_val:   (757, 26)
X_test:  (946, 26)

y_train: (3027,)
y_val:   (757,)
y_test:  (946,)


In [ ]:
distribution_summary = pd.DataFrame({
    "train_%": (y_train.value_counts(normalize=True) * 100).round(2),
    "val_%":   (y_val.value_counts(normalize=True) * 100).round(2),
    "test_%":  (y_test.value_counts(normalize=True) * 100).round(2)
}).sort_index()

display(distribution_summary)

,train_%,val_%,test_%
is_canceled,,,
0,64.52,64.6,64.59
1,35.48,35.4,35.41


## 11. Identificació del tipus de variables

Per construir el pipeline de preprocessament, es distingeixen les variables numèriques de les categòriques, ja que cadascun d’aquests grups requerirà transformacions diferents.

In [ ]:
categorical_features_auto = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numerical_features_auto = X_train.select_dtypes(include=["number"]).columns.tolist()

print("Variables categòriques detectades automàticament:")
print(categorical_features_auto)
print()
print("Variables numèriques detectades automàticament:")
print(numerical_features_auto)


Variables categòriques detectades automàticament:
['hotel', 'arrival_date_month', 'gender_account', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'customer_type', 'country_risk']

Variables numèriques detectades automàticament:
['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'has_agent', 'has_company', 'total_nights', 'total_guests', 'adr_per_guest', 'is_family_booking']


## 12. Construcció del pipeline de preprocessament

A continuació, es defineix un pipeline de preprocessament mitjançant `ColumnTransformer`, que permet aplicar transformacions diferenciades segons el tipus de variable.

Les variables numèriques es processen mitjançant imputació de la mediana i escalat estàndard, mentre que les variables categòriques es tracten amb imputació del valor més freqüent i codificació *one-hot*.

In [ ]:
# ==========================================================
# PIPELINE PER A MACHINE LEARNING
# ==========================================================

binary_features_ml = [
    "is_repeated_guest",
    "has_agent",
    "has_company",
    "is_family_booking"
]

categorical_features_ml = [
    "hotel",
    "arrival_date_year",
    "arrival_date_month",
    "gender_account",
    "meal",
    "country_risk",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "customer_type"
]

numerical_features_ml = [
    "lead_time",
    "arrival_date_week_number",
    "arrival_date_day_of_month",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "adr",
    "required_car_parking_spaces",
    "total_of_special_requests",
    "total_nights",
    "total_guests",
    "adr_per_guest"
]

print("Variables binàries ML:")
print(binary_features_ml)
print()
print("Variables categòriques ML:")
print(categorical_features_ml)
print()
print("Variables numèriques ML:")
print(numerical_features_ml)



Variables binàries ML:
['is_repeated_guest', 'has_agent', 'has_company', 'is_family_booking']

Variables categòriques ML:
['hotel', 'arrival_date_year', 'arrival_date_month', 'gender_account', 'meal', 'country_risk', 'market_segment', 'distribution_channel', 'reserved_room_type', 'customer_type']

Variables numèriques ML:
['lead_time', 'arrival_date_week_number', 'arrival_date_day_of_month', 'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_nights', 'total_guests', 'adr_per_guest']


In [ ]:
numeric_pipeline_ml = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

binary_pipeline_ml = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

categorical_pipeline_ml = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor_ml = ColumnTransformer([
    ("num", numeric_pipeline_ml, numerical_features_ml),
    ("bin", binary_pipeline_ml, binary_features_ml),
    ("cat", categorical_pipeline_ml, categorical_features_ml)
])


## 13. Aplicació del preprocessament

El preprocessador s’ajusta únicament amb el conjunt d’entrenament i posteriorment s’aplica als conjunts de validació i test. Aquesta estratègia evita la filtració d’informació i garanteix que les transformacions es basin exclusivament en dades disponibles durant l’entrenament.

In [ ]:
X_train_processed = preprocessor_ml.fit_transform(X_train)
X_val_processed = preprocessor_ml.transform(X_val)
X_test_processed = preprocessor_ml.transform(X_test)

print("Dimensions després del preprocessament ML:")
print(f"X_train_processed: {X_train_processed.shape}")
print(f"X_val_processed:   {X_val_processed.shape}")
print(f"X_test_processed:  {X_test_processed.shape}")


Dimensions després del preprocessament ML:
X_train_processed: (3027, 65)
X_val_processed:   (757, 65)
X_test_processed:  (946, 65)


## 14. Conversió dels conjunts preprocessats a estructures interpretables

Per facilitar la interpretació posterior dels resultats, els conjunts preprocessats es converteixen en `DataFrame` i s’hi associen els noms de les variables resultants després de la codificació *one-hot*.

In [ ]:
feature_names_ml = preprocessor_ml.get_feature_names_out()

X_train_df = pd.DataFrame(X_train_processed, columns=feature_names_ml)
X_val_df = pd.DataFrame(X_val_processed, columns=feature_names_ml)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names_ml)

display(X_train_df.head())
print("Nombre final de variables després del preprocessament ML:", len(feature_names_ml))


,num__lead_time,num__arrival_date_week_number,num__arrival_date_day_of_month,num__previous_cancellations,num__previous_bookings_not_canceled,num__adr,num__required_car_parking_spaces,num__total_of_special_requests,num__total_nights,num__total_guests,num__adr_per_guest,bin__is_repeated_guest,bin__has_agent,bin__has_company,bin__is_family_booking,cat__hotel_City Hotel,cat__hotel_Resort Hotel,cat__arrival_date_year_2015,cat__arrival_date_year_2016,cat__arrival_date_year_2017,cat__arrival_date_month_April,cat__arrival_date_month_August,cat__arrival_date_month_December,cat__arrival_date_month_February,cat__arrival_date_month_January,cat__arrival_date_month_July,cat__arrival_date_month_June,cat__arrival_date_month_March,cat__arrival_date_month_May,cat__arrival_date_month_November,cat__arrival_date_month_October,cat__arrival_date_month_September,cat__gender_account_Female,cat__gender_account_Male,cat__meal_BB,cat__meal_FB,cat__meal_HB,cat__meal_SC,cat__meal_Undefined,cat__country_risk_High,cat__country_risk_Low,cat__country_risk_Medium,cat__market_segment_Aviation,cat__market_segment_Complementary,cat__market_segment_Corporate,cat__market_segment_Direct,cat__market_segment_Groups,cat__market_segment_Offline TA/TO,cat__market_segment_Online TA,cat__distribution_channel_Corporate,cat__distribution_channel_Direct,cat__distribution_channel_GDS,cat__distribution_channel_TA/TO,cat__reserved_room_type_A,cat__reserved_room_type_B,cat__reserved_room_type_C,cat__reserved_room_type_D,cat__reserved_room_type_E,cat__reserved_room_type_F,cat__reserved_room_type_G,cat__reserved_room_type_H,cat__customer_type_Contract,cat__customer_type_Group,cat__customer_type_Transient,cat__customer_type_Transient-Party
0,0.577063,-0.008231,-1.692056,-0.099464,-0.103066,1.379032,-0.26384,0.509973,-0.156363,3.059620,-0.448008,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.807063,-0.008231,-0.898164,-0.099464,-0.103066,-0.394719,-0.26384,0.509973,-0.547854,0.037933,-0.478494,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.774184,0.569541,1.483513,-0.099464,-0.103066,-0.239105,-0.26384,0.509973,-0.939345,0.037933,-0.341853,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.874146,-0.369339,-1.578643,-0.099464,-0.103066,-0.032447,-0.26384,-0.729938,-0.547854,0.037933,-0.160390,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,1.353311,1.075091,0.122555,1.355023,-0.103066,-0.859081,-0.26384,-0.729938,-0.939345,0.037933,-0.886239,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


Nombre final de variables després del preprocessament ML: 65


In [ ]:
# ==========================================================
# VALIDACIÓ BÀSICA DELS CONJUNTS PREPROCESSATS PER A ML
# ==========================================================
print("=== Comprovacions generals ===")
print(f"Nuls a X_train_df: {X_train_df.isnull().sum().sum()}")
print(f"Nuls a X_val_df:   {X_val_df.isnull().sum().sum()}")
print(f"Nuls a X_test_df:  {X_test_df.isnull().sum().sum()}")

print(f"Infinits a X_train_df: {np.isinf(X_train_df.to_numpy()).sum()}")
print(f"Infinits a X_val_df:   {np.isinf(X_val_df.to_numpy()).sum()}")
print(f"Infinits a X_test_df:  {np.isinf(X_test_df.to_numpy()).sum()}")

num_cols_processed = [col for col in X_train_df.columns if col.startswith("num__")]
bin_cols_processed = [col for col in X_train_df.columns if col.startswith("bin__")]
cat_cols_processed = [col for col in X_train_df.columns if col.startswith("cat__")]

print("\n=== Nombre de columnes per bloc ===")
print(f"Numèriques escalades: {len(num_cols_processed)}")
print(f"Binàries:             {len(bin_cols_processed)}")
print(f"One-hot categòriques: {len(cat_cols_processed)}")

if num_cols_processed:
    print("\n=== Resum de les variables numèriques preprocessades ===")
    numeric_check = X_train_df[num_cols_processed].agg(["mean", "std", "min", "max"]).T
    display(numeric_check.head(15))

if bin_cols_processed:
    print("\n=== Comprovació de variables binàries ===")
    binary_values_ok = X_train_df[bin_cols_processed].isin([0, 1]).all().all()
    print(f"Totes les variables binàries només tenen valors 0/1? {binary_values_ok}")
    display(X_train_df[bin_cols_processed].describe().T)

=== Comprovacions generals ===
Nuls a X_train_df: 0
Nuls a X_val_df:   0
Nuls a X_test_df:  0
Infinits a X_train_df: 0
Infinits a X_val_df:   0
Infinits a X_test_df:  0

=== Nombre de columnes per bloc ===
Numèriques escalades: 11
Binàries:             4
One-hot categòriques: 50

=== Resum de les variables numèriques preprocessades ===


,mean,std,min,max
num__lead_time,-5.398904e-17,1.000165,-0.937101,5.023721
num__arrival_date_week_number,-9.389399e-18,1.000165,-1.885991,1.869528
num__arrival_date_day_of_month,-9.624134e-17,1.000165,-1.692056,1.710340
num__previous_cancellations,-2.640768e-18,1.000165,-0.099464,37.717212
num__previous_bookings_not_canceled,-8.802561e-18,1.000165,-0.103066,37.840555
num__adr,-3.990494e-17,1.000165,-2.099033,7.231606
num__required_car_parking_spaces,-1.173675e-18,1.000165,-0.263840,3.790182
num__total_of_special_requests,-1.877880e-17,1.000165,-0.729938,5.469619
num__total_nights,-1.114991e-17,1.000165,-1.330835,8.456430
num__total_guests,-2.816820e-17,1.000165,-2.983753,4.570463



=== Comprovació de variables binàries ===
Totes les variables binàries només tenen valors 0/1? True


,count,mean,std,min,25%,50%,75%,max
bin__is_repeated_guest,3027.0,0.034688,0.183018,0.0,0.0,0.0,0.0,1.0
bin__has_agent,3027.0,0.858606,0.348485,0.0,1.0,1.0,1.0,1.0
bin__has_company,3027.0,0.058474,0.234676,0.0,0.0,0.0,0.0,1.0
bin__is_family_booking,3027.0,0.084572,0.278290,0.0,0.0,0.0,0.0,1.0


## 15. Emmagatzematge i reutilització

Finalment, es guarden els conjunts preprocessats i el preprocessador entrenat, de manera que les mateixes transformacions puguin reutilitzar-se posteriorment tant en la fase de modelització com en un possible entorn de producció.

In [ ]:
ML_OUTPUT_DIR = RUNS_DIR / "dades_preprocessades_ml"
ML_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

X_train_df.to_csv(ML_OUTPUT_DIR / "X_train.csv", index=False)
X_val_df.to_csv(ML_OUTPUT_DIR / "X_val.csv", index=False)
X_test_df.to_csv(ML_OUTPUT_DIR / "X_test.csv", index=False)

pd.DataFrame({TARGET: y_train.reset_index(drop=True)}).to_csv(ML_OUTPUT_DIR / "y_train.csv", index=False)
pd.DataFrame({TARGET: y_val.reset_index(drop=True)}).to_csv(ML_OUTPUT_DIR / "y_val.csv", index=False)
pd.DataFrame({TARGET: y_test.reset_index(drop=True)}).to_csv(ML_OUTPUT_DIR / "y_test.csv", index=False)

joblib.dump(preprocessor_ml, ML_OUTPUT_DIR / "preprocessor.joblib")

print(f"Fitxers de ML desats a: {ML_OUTPUT_DIR.resolve()}")


Fitxers de ML desats a: /content/dades_preprocessades_ml


In [ ]:
# ==========================================================
# DATASET PER A CLUSTERING
# ==========================================================

CLUSTERING_OUTPUT_DIR = RUNS_DIR / "dades_preprocessades_clustering"
CLUSTERING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

clustering_drop_columns = [
    # TARGET,
    "arrival_date_week_number",
    "required_car_parking_spaces",
    "adr"
]

clustering_df = df.drop(
    columns=[col for col in clustering_drop_columns if col in df.columns],
    errors="ignore"
).copy()

clustering_df.to_csv(CLUSTERING_OUTPUT_DIR / "clustering_dataset.csv", index=False)

print(f"Dimensions del dataset per a clustering: {clustering_df.shape}")
display(clustering_df.head())
print(f"Fitxer de clustering desat a: {CLUSTERING_OUTPUT_DIR.resolve()}")


Dimensions del dataset per a clustering: (4730, 24)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_day_of_month,gender_account,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,customer_type,total_of_special_requests,has_agent,has_company,total_nights,total_guests,adr_per_guest,is_family_booking,country_risk
0,Resort Hotel,0,203,2016,December,2,Female,BB,GBR,Direct,Direct,0,0,0,F,Transient,0,1,0,7,2.0,33.40,0,Low
1,City Hotel,1,82,2015,July,16,Female,BB,PRT,Online TA,TA/TO,0,0,0,A,Transient,0,1,0,3,2.0,38.25,0,High
2,City Hotel,0,25,2016,December,27,Female,BB,BRA,Offline TA/TO,TA/TO,0,0,0,A,Transient-Party,1,1,0,3,3.0,20.00,0,Medium
3,City Hotel,0,1,2016,March,9,Female,BB,Other,Online TA,TA/TO,0,0,0,A,Transient-Party,0,1,0,1,1.0,95.00,0,Low
4,City Hotel,0,70,2017,April,16,Female,SC,GBR,Online TA,TA/TO,0,0,0,A,Transient,0,1,0,4,2.0,54.00,0,Low


Fitxer de clustering desat a: /content/dades_preprocessades_clustering
